# 0.1 Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# 0.2 Clone GitHub Repo

In [ ]:
REPO_URL = "https://github.com/heyitzrizki/3d-cancer-cell-bioimage-ai-pipeline.git"
REPO_ROOT = "/content/3d-cancer-cell-bioimage-ai-pipeline"
!git clone "$REPO_URL" "$REPO_ROOT"
%cd "$REPO_ROOT"

# 0.3 Install Requirements

In [ ]:
%pip install -q -r requirements-colab.txt
%pip install -q -e . --no-deps

# 0.4 Set Paths

In [ ]:
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/self learning/Self Project/3d-cancer-cell-bioimage-ai-pipeline-data")
DATASET_DIR = DATA_ROOT / "data" / "raw" / "Fluo-C3DL-MDA231"
SECONDARY_DATASET_DIR = DATA_ROOT / "data" / "raw" / "Fluo-C3DH-A549"
OUTPUT_DIR = DATA_ROOT / "reports" / "milestone_5"
CHECKPOINT_DIR = DATA_ROOT / "checkpoints" / "milestone_5"
EVAL_DIR = DATA_ROOT / "reports" / "milestone_5_eval"

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"SECONDARY_DATASET_DIR: {SECONDARY_DATASET_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"CHECKPOINT_DIR: {CHECKPOINT_DIR}")

# 1.1 Verify Dataset Location

In [ ]:
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Dataset not found: {DATASET_DIR}")
if not (DATASET_DIR / "01").is_dir():
    raise FileNotFoundError(f"Sequence folder not found: {DATASET_DIR / '01'}")
if not (DATASET_DIR / "01_GT" / "SEG").is_dir():
    raise FileNotFoundError(f"Official SEG folder not found: {DATASET_DIR / '01_GT' / 'SEG'}")
print("Main dataset OK:", DATASET_DIR)
print("Secondary dataset exists:", SECONDARY_DATASET_DIR.is_dir(), SECONDARY_DATASET_DIR)

# 1.2 Inspect Available GT Masks

In [ ]:
for sequence in ("01", "02"):
    gt_files = sorted((DATASET_DIR / f"{sequence}_GT" / "SEG").glob("*.tif*"))
    print(f"Sequence {sequence}: {len(gt_files)} official SEG files")

# 2.1 Train 2D U-Net

In [ ]:
!python scripts/train_unet2d.py --dataset-dir "{str(DATASET_DIR)}" --output-dir "{str(OUTPUT_DIR)}" --checkpoint-dir "{str(CHECKPOINT_DIR)}" --sequence-ids 01 02 --epochs 10 --batch-size 2 --device auto

# 2.2 View Training History

In [ ]:
import pandas as pd

history_path = OUTPUT_DIR / "metrics" / "unet_2d_training_history.csv"
if not history_path.exists():
    raise FileNotFoundError(
        f"Training history not found: {history_path}\n"
        "Run section 2.1 Train 2D U-Net first and make sure it finishes without errors."
    )

history = pd.read_csv(history_path)
history

# 3.1 Evaluate Best Checkpoint

In [ ]:
CHECKPOINT_PATH = CHECKPOINT_DIR / "unet_2d_best.pt"
!python scripts/evaluate_unet2d.py --dataset-dir "{str(DATASET_DIR)}" --checkpoint-path "{str(CHECKPOINT_PATH)}" --output-dir "{str(EVAL_DIR)}" --sequence-ids 01 02 --device auto

# 3.2 Visualize Prediction Overlay

In [ ]:
from IPython.display import Image, display

overlay_path = OUTPUT_DIR / "figures" / "unet_2d_prediction_overlay.png"
if not overlay_path.exists():
    raise FileNotFoundError(
        f"Prediction overlay not found: {overlay_path}\n"
        "Run section 2.1 Train 2D U-Net first and make sure it finishes without errors."
    )

display(Image(filename=str(overlay_path)))

# 4.1 Save Results to Drive

In [ ]:
print(f"Training outputs already saved to: {OUTPUT_DIR}")
print(f"Checkpoint already saved to: {CHECKPOINT_DIR}")
print(f"Evaluation outputs already saved to: {EVAL_DIR}")

# 5.1 Summary and Next Steps

Review validation overlays and sparse-plane metrics before tuning the model or extending the pipeline to patch-based 3D U-Net training.